## Get ASpop for 3D scoops of 3D reference objects

## Install and import libraries

In [9]:
%pip install requests pandas

import requests
import pandas as pd
from  io import StringIO
import warnings

Note: you may need to restart the kernel to use updated packages.


In [10]:
warnings.filterwarnings('ignore')

## Get data

In [11]:
# get 3d reference object crosswalks
url = 'https://cdn.humanatlas.io/digital-objects/ref-organ/asct-b-3d-models-crosswalk/v1.8/assets/asct-b-3d-models-crosswalk.csv'

df_crosswalk = pd.read_csv(url, skiprows=10)

# adjust columns for later match with ASpop data
df_crosswalk = df_crosswalk.rename(columns={
  'representation_of':'as'
})
# df_crosswalk['as'] = 'http://purl.obolibrary.org/obo/' + df_crosswalk['as'].replace(':','_')

df_crosswalk

,anatomical_structure_of,source_spatial_entity,node_name,label,OntologyID,as,node_type,glb file of single organs,Ref/1,Ref/1/ID
0,-,-,VH_F,-,-,-,organizational,3d-vh-f-united,NaN,NaN
1,-,#VHFemaleOrgans,VH_F_integumentary_system,integumentary system layer,UBERON:0013754,http://purl.obolibrary.org/obo/UBERON_0013754,organizational,3d-vh-f-united,NaN,NaN
2,#VHFSkinV1.2,#VHFemaleOrgans,VH_F_skin,skin of body,UBERON:0002097,http://purl.obolibrary.org/obo/UBERON_0002097,mesh,VH_F_Skin,NaN,NaN
3,-,-,VH_F_mammary_gland,-,-,-,organizational,-,NaN,NaN
4,#VHFLeftMammaryGland,#VHFemaleOrgans,VH_F_mammary_gland_L,Left mammary gland,FMA:57991,http://purl.org/sig/ont/fma/fma57991,organizational,3d-vh-f-mammary-gland-l,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2175,#VHMVertebrae,#VHMaleOrgans,VH_M_lumbar_vertebra_1,lumbar vertebra 1,UBERON:0004617,http://purl.obolibrary.org/obo/UBERON_0004617,mesh,VH_M_Vertebrae,NaN,NaN
2176,#VHMVertebrae,#VHMaleOrgans,VH_M_lumbar_vertebra_2,lumbar vertebra 2,UBERON:0004618,http://purl.obolibrary.org/obo/UBERON_0004618,mesh,VH_M_Vertebrae,NaN,NaN
2177,#VHMVertebrae,#VHMaleOrgans,VH_M_lumbar_vertebra_3,lumbar vertebra 3,UBERON:0004619,http://purl.obolibrary.org/obo/UBERON_0004619,mesh,VH_M_Vertebrae,NaN,NaN
2178,#VHMVertebrae,#VHMaleOrgans,VH_M_lumbar_vertebra_4,lumbar vertebra 4,UBERON:0004620,http://purl.obolibrary.org/obo/UBERON_0004620,mesh,VH_M_Vertebrae,NaN,NaN


In [12]:
# get ASpop
url = 'https://apps.humanatlas.io/api/grlc/hra-pop/cell_types_in_anatomical_structurescts_per_as'

headers = {
  'accept':'text/csv'
}

response = requests.get(url, headers=headers)

#  Convert the response content to a StringIO object
csv_data = StringIO(response.text)

# Read the CSV data into a DataFrame
df_as_pop = pd.read_csv(csv_data)
df_as_pop

,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count
0,large intestine,http://purl.obolibrary.org/obo/UBERON_0001052,rectum,Female,celltypist,sc_transcriptomics,https://purl.org/ccf/ASCTB-TEMP_colonocyte,Colonocyte,1.205,0.147653,3
1,large intestine,http://purl.obolibrary.org/obo/UBERON_0001052,rectum,Female,celltypist,sc_transcriptomics,https://purl.org/ccf/ASCTB-TEMP_iga-plasma-cell,IgA plasma cell,1.182,0.144835,3
2,large intestine,http://purl.obolibrary.org/obo/UBERON_0001052,rectum,Female,celltypist,sc_transcriptomics,https://purl.org/ccf/ASCTB-TEMP_best4-epithelial,BEST4+ epithelial,0.699,0.085651,3
3,large intestine,http://purl.obolibrary.org/obo/UBERON_0001052,rectum,Female,celltypist,sc_transcriptomics,https://purl.org/ccf/ASCTB-TEMP_activated-cd4-t,Activated CD4 T,0.690,0.084548,3
4,large intestine,http://purl.obolibrary.org/obo/UBERON_0001052,rectum,Female,celltypist,sc_transcriptomics,https://purl.org/ccf/ASCTB-TEMP_ta,TA,0.540,0.066168,3
...,...,...,...,...,...,...,...,...,...,...,...
8891,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_0000097,Mast Cell,15322.464,0.024702,1
8892,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,http://purl.obolibrary.org/obo/CL_4033039,CD8+ T Cell,3691.176,0.005951,1
8893,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,https://purl.org/ccf/ASCTB-TEMP_lymphatic-endo...,Lymphatic Endothelial (and some immune cells),1753.956,0.002828,1
8894,lung,http://purl.org/sig/ont/fma/fma7508,Left posterior basal segmental bronchus,Male,sc_proteomics,sc_proteomics,https://purl.org/ccf/ASCTB-TEMP_basal-epitheli...,Basal Epithelial Cell,970.104,0.001564,1


## Preprocess to only keep ASpop for male heart, left kidney, prostate, small intestine

In [13]:
# filter
organs_of_interest = {
  'Left kidney': 'http://purl.obolibrary.org/obo/UBERON_0004538',  # left kidney
  'heart': 'http://purl.obolibrary.org/obo/UBERON_0000948',  # heart
  'prostate': 'http://purl.obolibrary.org/obo/UBERON_0000079',  # prostate
  'small intestine': 'http://purl.obolibrary.org/obo/UBERON_0002108',  # small intestine
}

sex = 'Male'

df_as_filtered = df_as_pop[(df_as_pop['organ'].isin(organs_of_interest.keys())) & (df_as_pop['sex'] == sex)]
df_as_filtered

,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count
4191,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell,157.950,0.319973,2
4192,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002341,basal cell of prostate epithelium,128.700,0.260718,2
4193,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000625,"CD8-positive, alpha-beta T cell",56.862,0.115190,2
4194,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002340,luminal cell of prostate epithelium,38.688,0.078374,2
4195,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000158,club cell,32.812,0.066470,2
...,...,...,...,...,...,...,...,...,...,...,...
7412,small intestine,http://purl.org/sig/ont/fma/fma7206,superior part of duodenum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000131,gut endothelial cell,0.055,0.001580,1
7413,small intestine,http://purl.org/sig/ont/fma/fma7206,superior part of duodenum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0009080,intestinal tuft cell,0.055,0.001580,1
7414,small intestine,http://purl.org/sig/ont/fma/fma7206,superior part of duodenum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000786,plasma cell,0.055,0.001580,1
7415,small intestine,http://purl.org/sig/ont/fma/fma7206,superior part of duodenum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000775,neutrophil,0.022,0.000632,1


In [14]:
# join with crosswalk to get node

df_result = df_as_filtered.merge(df_crosswalk, on='as', how='inner')
df_result

,organ,as,as_label,sex,tool,modality,cell_id,cell_label,cell_count,cell_percentage,dataset_count,anatomical_structure_of,source_spatial_entity,node_name,label,OntologyID,node_type,glb file of single organs,Ref/1,Ref/1/ID
0,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000066,epithelial cell,157.950,0.319973,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_vesicle,seminal vesicle,UBERON:0000998,mesh,VH_M_Prostate,NaN,NaN
1,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002341,basal cell of prostate epithelium,128.700,0.260718,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_vesicle,seminal vesicle,UBERON:0000998,mesh,VH_M_Prostate,NaN,NaN
2,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000625,"CD8-positive, alpha-beta T cell",56.862,0.115190,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_vesicle,seminal vesicle,UBERON:0000998,mesh,VH_M_Prostate,NaN,NaN
3,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0002340,luminal cell of prostate epithelium,38.688,0.078374,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_vesicle,seminal vesicle,UBERON:0000998,mesh,VH_M_Prostate,NaN,NaN
4,prostate,http://purl.obolibrary.org/obo/UBERON_0000998,seminal vesicle,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000158,club cell,32.812,0.066470,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_vesicle,seminal vesicle,UBERON:0000998,mesh,VH_M_Prostate,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6460,prostate,http://purl.org/sig/ont/fma/fma19719,Verumontanum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000192,smooth muscle cell,0.369,0.019435,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_colliculus,Verumontanum,FMA:19719,surface,VH_M_Prostate,NaN,NaN
6461,prostate,http://purl.org/sig/ont/fma/fma19719,Verumontanum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000057,fibroblast,0.283,0.014906,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_colliculus,Verumontanum,FMA:19719,surface,VH_M_Prostate,NaN,NaN
6462,prostate,http://purl.org/sig/ont/fma/fma19719,Verumontanum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000038,erythroid progenitor cell,0.105,0.005530,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_colliculus,Verumontanum,FMA:19719,surface,VH_M_Prostate,NaN,NaN
6463,prostate,http://purl.org/sig/ont/fma/fma19719,Verumontanum,Male,popv,sc_transcriptomics,http://purl.obolibrary.org/obo/CL_0000576,monocyte,0.091,0.004793,2,#VHMProstate,#VHMaleOrgans,VH_M_seminal_colliculus,Verumontanum,FMA:19719,surface,VH_M_Prostate,NaN,NaN


## Export to CSV

In [15]:
df_result.to_csv('output/as-pop-scoops.csv', index=True)